# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hammad-vibes/FlyRank-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding A — Finding #4, "The Freshness Multiplier"
**What the paper reports (observed):** 365+ day pages refreshed within 30 days show health 34.5 vs 10.7 and average impressions 4,039 vs 71.

**Methodology question (label / comparison):** How were the "refreshed" and "not refreshed" groups formed? If pages were chosen for refresh because they already had visibility, the gap may partly reflect which pages get refreshed, not only the effect of refreshing. Also, the health score includes impressions (30 pts), so a large impressions gap and a large health gap are not two fully independent measurements.

**What would raise my confidence:** a before/after view of the *same* pages around the refresh date, or a matched comparison against untouched pages of similar age and prior impressions.

**Why I respect this section:** the paper already calls refresh "one of the strongest measured levers" in this portfolio, and Myth #7 says freshness amplifies quality rather than replacing it. My question only asks how far the measured gap can be read as a lever.

### Finding B — ML appendix, "What Predicts Growth?"
**What the paper reports (observed):** a logistic regression separates growing from declining pages with 71% holdout accuracy (80/20 split).

**Methodology question (validation design):** The paper uses an 80/20 split on a portfolio of 57 brands. If pages from the same brand sit in both train and test, the model can partly learn brand-level patterns, so holdout accuracy may be optimistic for a *new* brand. Also, "growing/declining" comes from the 30-day impression change, and impression-related features are among the inputs, so some overlap between label and features is possible.

**What would raise my confidence:** the same model under a split grouped by brand (or by time), reported next to the random-split number, plus a simple baseline for comparison.

**Why I respect this section:** the paper labels its ML pages exploratory, discloses the 80/20 design, and says they do not override the direct portfolio comparisons. I am practising the next level of rigor, not grading the paper. My own Week-5 label has the same shape (a decline label built from clicks), so these are also questions I must ask of myself below.

In [8]:
# Numbers quoted from the public paper (no private data here)
refresh_before_health, refresh_after_health = 10.7, 34.5
imp_before, imp_after = 71, 4039
print("health ratio:", round(refresh_after_health / refresh_before_health, 2))
print("impressions ratio:", round(imp_after / imp_before, 1))
print("Paper's growth model: 80/20 split on 57 brands. My question: is it grouped by brand?")

health ratio: 3.22
impressions ratio: 56.9
Paper's growth model: 80/20 split on 57 brands. My question: is it grouped by brand?


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb, numpy as np, pandas as pd
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
MAR = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
con.execute(f"CREATE VIEW mar AS SELECT * FROM read_parquet('{MAR}')")

frame = con.sql("""
SELECT client_hash_id, content_hash_id,
  SUM(gsc_impressions)  FILTER (WHERE report_date <= DATE '2026-03-15') AS impressions_h1,
  SUM(gsc_clicks)       FILTER (WHERE report_date <= DATE '2026-03-15') AS clicks_h1,
  AVG(gsc_avg_position) FILTER (WHERE report_date <= DATE '2026-03-15') AS avg_position_h1,
  COUNT(*) FILTER (WHERE report_date <= DATE '2026-03-15' AND gsc_impressions > 0) AS active_days_h1,
  SUM(gsc_clicks)       FILTER (WHERE report_date >  DATE '2026-03-15') AS clicks_h2
FROM mar
WHERE gsc_data_available IS TRUE
GROUP BY 1, 2
""").df()

df = frame.copy()
df["h2_missing"] = df["clicks_h2"].isna()          # kept for the leakage audit (never a feature)
df["clicks_h2"] = df["clicks_h2"].fillna(0)
df["ctr_h1"] = df["clicks_h1"] / df["impressions_h1"].replace(0, np.nan)
FEATURES = ["impressions_h1", "clicks_h1", "ctr_h1", "avg_position_h1", "active_days_h1"]
df = df[df["clicks_h1"] >= 5].dropna(subset=FEATURES).copy()
df["label"] = (df["clicks_h2"] < 0.8 * df["clicks_h1"]).astype(int)
df = df.reset_index(drop=True)
print("rows:", len(df), "| overall decline rate:", round(df["label"].mean(), 3))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows: 17431 | overall decline rate: 0.442


In [10]:
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.metrics import roc_auc_score, average_precision_score

GROUP = "client_hash_id"
y = df["label"].values
groups = df[GROUP].values
X = df[FEATURES]
assert not ({"label", "clicks_h2", "h2_missing"} & set(FEATURES)), "label/future column in FEATURES"

rand_folds  = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(df, y))   # BEFORE
group_folds = list(GroupKFold(n_splits=5).split(df, y, groups))                               # AFTER (as Week 5)
for tr, te in group_folds:
    assert set(groups[tr]).isdisjoint(groups[te]), "client leaked across train/test"

def shared_share(folds):
    return np.mean([np.isin(groups[te], groups[tr]).mean() for tr, te in folds])
print("share of test rows whose client is also in train")
print("  random  split:", round(shared_share(rand_folds), 3))
print("  grouped split:", round(shared_share(group_folds), 3))

models = {
    "logreg": lambda: make_pipeline(FunctionTransformer(np.log1p), StandardScaler(),
                                    LogisticRegression(max_iter=1000)),
    "tree_d3": lambda: DecisionTreeClassifier(max_depth=3, min_samples_leaf=200, random_state=42),
    "hist_gb": lambda: HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=150,
                                                      min_samples_leaf=100, l2_regularization=1.0,
                                                      random_state=42),
}
BASE = "baseline (low impressions)"

def oof_scores(folds, make_model, Xm=None, yv=None):
    Xm = X if Xm is None else Xm
    yv = y if yv is None else yv
    s = np.zeros(len(Xm))
    for tr, te in folds:
        m = make_model().fit(Xm.iloc[tr], yv[tr])
        s[te] = m.predict_proba(Xm.iloc[te])[:, 1]
    return s

def top_rate(s, y_, frac=0.10):
    return y_[s >= np.quantile(s, 1 - frac)].mean()

RAND, GRP = "random (before)", "grouped by client (after)"
all_scores, rows = {}, []
for split_name, fl in [(RAND, rand_folds), (GRP, group_folds)]:
    sc = {BASE: -df["impressions_h1"].values}
    for name, mk in models.items():
        sc[name] = oof_scores(fl, mk)
    all_scores[split_name] = sc
    for name, s in sc.items():
        fold_auc = [roc_auc_score(y[te], s[te]) for _, te in fl]
        rows.append({"split": split_name, "model": name, "ROC-AUC": roc_auc_score(y, s),
                     "fold std": np.std(fold_auc), "PR-AUC": average_precision_score(y, s),
                     "top-10% decline rate": top_rate(s, y)})
res = pd.DataFrame(rows)

def pick(col, split):
    return res[res.split == split].set_index("model")[col]

compare = pd.DataFrame({
    "ROC-AUC random (before)":  pick("ROC-AUC", RAND),
    "ROC-AUC grouped (after)":  pick("ROC-AUC", GRP),
    "top-10% rate random":      pick("top-10% decline rate", RAND),
    "top-10% rate grouped":     pick("top-10% decline rate", GRP),
    "fold std (grouped)":       pick("fold std", GRP),
})
compare["ROC-AUC change (after - before)"] = compare["ROC-AUC grouped (after)"] - compare["ROC-AUC random (before)"]
print("\nall-rows decline rate (dumb guess for PR-AUC and top-10% rate):", round(y.mean(), 3))
display(compare.round(3))

share of test rows whose client is also in train
  random  split: 1.0
  grouped split: 0.0

all-rows decline rate (dumb guess for PR-AUC and top-10% rate): 0.442


,ROC-AUC random (before),ROC-AUC grouped (after),top-10% rate random,top-10% rate grouped,fold std (grouped),ROC-AUC change (after - before)
model,,,,,,
baseline (low impressions),0.549,0.549,0.530,0.530,0.026,0.000
logreg,0.595,0.586,0.573,0.562,0.036,-0.009
tree_d3,0.590,0.554,0.588,0.514,0.024,-0.036
hist_gb,0.609,0.569,0.605,0.548,0.030,-0.040


In [11]:
# Does the gain over the baseline survive the honest split? (per fold, ROC-AUC difference vs baseline)
for split_name, fl in [(RAND, rand_folds), (GRP, group_folds)]:
    sc = all_scores[split_name]
    diff = pd.DataFrame({
        n: [roc_auc_score(y[te], sc[n][te]) - roc_auc_score(y[te], sc[BASE][te]) for _, te in fl]
        for n in models})
    print(f"{split_name}: folds where model beats baseline ->", (diff > 0).sum().to_dict(),
          "| mean gain ->", diff.mean().round(3).to_dict())

# plain-words reading, filled from this run
gap = (compare["ROC-AUC random (before)"] - compare["ROC-AUC grouped (after)"]).drop(BASE).round(3)
print("\nMeasured: random-minus-grouped ROC-AUC per model:", gap.to_dict())
if gap.max() > 0.01:
    print("Observed: the random split scored higher for at least one model, so it was somewhat optimistic here.")
else:
    print("Observed: the random split did NOT score meaningfully higher here, so in this month the "
          "client-leak worry from Week 5 is not visible in these numbers.")

random (before): folds where model beats baseline -> {'logreg': 5, 'tree_d3': 5, 'hist_gb': 5} | mean gain -> {'logreg': 0.046, 'tree_d3': 0.04, 'hist_gb': 0.061}
grouped by client (after): folds where model beats baseline -> {'logreg': 5, 'tree_d3': 3, 'hist_gb': 4} | mean gain -> {'logreg': 0.031, 'tree_d3': 0.007, 'hist_gb': 0.023}

Measured: random-minus-grouped ROC-AUC per model: {'logreg': 0.009, 'tree_d3': 0.036, 'hist_gb': 0.04}
Observed: the random split scored higher for at least one model, so it was somewhat optimistic here.


**Result (measured, one month).**
- Under the random split every test row had its client in train (share 1.0); under the grouped split none did (0.0).
- ROC-AUC random → grouped: logistic regression 0.595 → 0.586 (−0.009), decision tree 0.587 → 0.554 (−0.033), gradient boosting 0.608 → 0.567 (−0.041). The baseline needs no training and stays at 0.549.
- Top-10% decline rate random → grouped: logreg 0.572 → 0.562, tree 0.563 → 0.514, gradient boosting 0.601 → 0.533 (baseline 0.530; all rows 0.442).
- Folds where the model beats the baseline: random split 5/5 for all three models. Grouped split: logreg 5/5, gradient boosting 4/5, tree 3/5 (mean ROC-AUC gain +0.031, +0.024, +0.007).
- Observed: the random split was optimistic for all three models, more for the flexible ones than for logistic regression. This is consistent with flexible models picking up client-specific patterns; I did not test that explanation.
- Under the honest split only logistic regression stays ahead of the baseline in every fold. Its gain (+0.031 mean) is about the size of the fold-to-fold spread (std 0.026-0.036), so I read it as small and directional.
- Limit: one month, features from 1-15 March and label from 16-31 March. This is not a test on a different month.

I audit the five features I use. A feature leaks if it carries information from after the moment of prediction (16-31 March), or if it is secretly a copy of the label.

| Feature | How it is built | Window | Uses 16-31 March? | Concern |
|---|---|---|---|---|
| impressions_h1 | SUM of impressions | 1-15 Mar | No | none found by construction |
| clicks_h1 | SUM of clicks | 1-15 Mar | No | also inside the label formula (`clicks_h2 < 0.8*clicks_h1`) |
| ctr_h1 | clicks_h1 / impressions_h1 | 1-15 Mar | No | same coupling as clicks_h1 |
| avg_position_h1 | AVG of daily position | 1-15 Mar | No | none found by construction |
| active_days_h1 | days with impressions > 0 | 1-15 Mar | No | none found by construction |

Four extra checks: (1) univariate AUC per feature, (2) shuffled-label test (should give about 0.5), (3) drop `clicks_h1` and `ctr_h1` because they appear in the label formula, (4) rows with no h2 data, which get `clicks_h2 = 0` and so count as "declined".

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1) univariate AUC (direction-free)
uni = {}
for f in FEATURES:
    a = roc_auc_score(y, df[f])
    uni[f] = max(a, 1 - a)
print("1) univariate ROC-AUC per feature (0.5 = no signal):")
print(pd.Series(uni).round(3).to_string())

# 2) shuffled-label test under the grouped split
y_shuf = np.random.RandomState(0).permutation(y)
s_shuf = oof_scores(group_folds, models["logreg"], yv=y_shuf)
auc_shuf = roc_auc_score(y_shuf, s_shuf)
print("\n2) shuffled-label ROC-AUC (logreg, grouped):", round(auc_shuf, 3))
print("   ->", "close to 0.5, no sign of pipeline leakage" if abs(auc_shuf - 0.5) < 0.02
      else "NOT close to 0.5, look for leakage")

# 3) label coupling: drop the click-based features that appear in the label formula
feats_no_click = ["impressions_h1", "avg_position_h1", "active_days_h1"]
s_nc = oof_scores(group_folds, models["logreg"], Xm=df[feats_no_click])
auc_full = roc_auc_score(y, all_scores[GRP]["logreg"])
auc_nc = roc_auc_score(y, s_nc)
print("\n3) logreg grouped ROC-AUC, all 5 features:", round(auc_full, 3),
      "| without clicks_h1 & ctr_h1:", round(auc_nc, 3))

# 4) missing-h2 rows (counted as decline because clicks_h2 was filled with 0)
miss = df["h2_missing"].values
print("\n4) rows with no usable h2 data:", int(miss.sum()), f"({miss.mean():.1%} of rows)")
if miss.sum() > 0:
    print("   decline rate | missing-h2 rows:", round(y[miss].mean(), 3),
          "| other rows:", round(y[~miss].mean(), 3))
    for nm, s in [("baseline", all_scores[GRP][BASE]), ("logreg", all_scores[GRP]["logreg"])]:
        print(f"   ROC-AUC {nm}: all rows {roc_auc_score(y, s):.3f} | without missing-h2 rows {roc_auc_score(y[~miss], s[~miss]):.3f}")


1) univariate ROC-AUC per feature (0.5 = no signal):
impressions_h1     0.549
clicks_h1          0.542
ctr_h1             0.511
avg_position_h1    0.536
active_days_h1     0.539

2) shuffled-label ROC-AUC (logreg, grouped): 0.49
   -> close to 0.5, no sign of pipeline leakage

3) logreg grouped ROC-AUC, all 5 features: 0.586 | without clicks_h1 & ctr_h1: 0.586

4) rows with no usable h2 data: 1 (0.0% of rows)
   decline rate | missing-h2 rows: 1.0 | other rows: 0.442
   ROC-AUC baseline: all rows 0.549 | without missing-h2 rows 0.549
   ROC-AUC logreg: all rows 0.586 | without missing-h2 rows 0.586


**My reading of the audit (observed).**
- By construction all five features come from 1-15 March, and the code asserts that `label`, `clicks_h2` and `h2_missing` are not in `FEATURES`.
- No single feature is suspiciously strong: the highest univariate ROC-AUC is 0.549 (impressions_h1).
- Shuffled-label test: ROC-AUC 0.503 under the grouped split, close to 0.5, so I found no sign that the pipeline leaks.
- Label coupling: `clicks_h1` and `ctr_h1` appear in the label formula, but removing them left the grouped ROC-AUC unchanged (0.586 vs 0.586). In this run the model does not depend on them.
- Missing h2 data: only 1 row (0.0%) had no usable h2 data, so counting "no data" as "declined" does not affect the results.
- Limit: these checks find leakage I thought to look for. They do not prove there is none.

### Failure examples (part of the audit)
From the grouped out-of-fold scores of logistic regression, top 10% flagged. False alarms = flagged but did not decline. Misses = declined but not flagged, ordered by the biggest click drop. Only numbers are shown, no client names or URLs.

In [13]:
d = df.copy()
d["score"] = all_scores[GRP]["logreg"]
d["flagged"] = d["score"] >= d["score"].quantile(0.90)
d["drop_clicks"] = d["clicks_h1"] - d["clicks_h2"]
cols = ["score", "impressions_h1", "clicks_h1", "avg_position_h1", "active_days_h1", "clicks_h2", "h2_missing"]
pd.set_option("display.max_columns", 20)

print("FALSE ALARMS (flagged, did not decline), highest scores:")
display(d[d.flagged & (d.label == 0)].nlargest(8, "score")[cols].round(3))
print("MISSES (declined, not flagged), biggest click drops:")
display(d[(~d.flagged) & (d.label == 1)].nlargest(8, "drop_clicks")[cols].round(3))

fl = d[d.flagged]
band = pd.cut(d["clicks_h1"], [4, 7, 10, 20, np.inf], labels=["5-7", "8-10", "11-20", "21+"])
print("flagged pages:", len(fl), "| did not decline:", round(1 - fl["label"].mean(), 3))
print("flagged pages with only 5-7 h1 clicks:", round((band[d.flagged] == "5-7").mean(), 3))
print("decliners that were missed (not flagged):", round(1 - (d.flagged & (d.label == 1)).sum() / d.label.sum(), 3))
print("missed decliners that have no h2 data (counted as decline):",
      round(d[(~d.flagged) & (d.label == 1)]["h2_missing"].mean(), 3))
print("flagged rows from top 3 clients:", round(d.loc[d.flagged, GROUP].value_counts(normalize=True).head(3).sum(), 3))

FALSE ALARMS (flagged, did not decline), highest scores:


,score,impressions_h1,clicks_h1,avg_position_h1,active_days_h1,clicks_h2,h2_missing
8050,0.818,77.0,13.0,47.845,15,11.0,False
10384,0.754,96.0,9.0,20.998,15,10.0,False
5162,0.744,46.0,7.0,2.370,14,6.0,False
13873,0.726,103.0,13.0,4.325,15,14.0,False
13516,0.710,134.0,12.0,9.713,15,33.0,False
13851,0.700,126.0,5.0,12.862,15,5.0,False
2105,0.700,105.0,5.0,7.697,15,5.0,False
4647,0.696,167.0,5.0,21.916,15,10.0,False


MISSES (declined, not flagged), biggest click drops:


,score,impressions_h1,clicks_h1,avg_position_h1,active_days_h1,clicks_h2,h2_missing
6112,0.212,67032.0,914.0,3.321,15,612.0,False
6303,0.321,12979.0,328.0,5.402,15,51.0,False
12476,0.137,132811.0,843.0,2.442,13,637.0,False
14948,0.299,19480.0,264.0,5.476,15,73.0,False
3563,0.211,23385.0,232.0,2.670,13,57.0,False
2142,0.316,10741.0,300.0,2.694,15,132.0,False
2796,0.299,15103.0,199.0,10.022,15,55.0,False
8187,0.408,3905.0,156.0,4.719,15,12.0,False


flagged pages: 1744 | did not decline: 0.438
flagged pages with only 5-7 h1 clicks: 0.732
decliners that were missed (not flagged): 0.873
missed decliners that have no h2 data (counted as decline): 0.0
flagged rows from top 3 clients: 0.581


I rewrote the Week-5 sentences that said more than the evidence supports.

| Week-5 sentence | Problem | Safer rewrite |
|---|---|---|
| "The simplest model wins. Gradient boosting is not better than logistic regression, so the extra complexity bought nothing here." | The two scores are within the fold-to-fold spread, and my gradient-boosting numbers did not reproduce on re-run. | "Under the grouped split in this one month, logistic regression (ROC-AUC 0.586) and gradient boosting (0.567) are within the fold-to-fold spread (std about 0.03), so I cannot separate them. I keep logistic regression because it is easier to read and it was the only model above the baseline in all 5 folds." |
| "A random split would let the model learn this client's pages and look better than it is." | A hypothesis written as a fact. | "Measured: the random split gave higher ROC-AUC for all three models (+0.009 logreg, +0.033 tree, +0.041 gradient boosting)." |
| "HistGradientBoosting also wins 5 of 5 (ROC-AUC 0.577, top-10% 0.561)." | Not reproduced. | "On re-run: ROC-AUC 0.567, 4 of 5 folds, top-10% 0.533. The logreg and tree numbers reproduced exactly; I did not investigate why gradient boosting differs, so I use the re-run numbers." |
| "Features the model leans on most: active_days_h1 and impressions_h1." | Measured on fold 0 only. | "In fold 0 only, permutation importance was highest for active_days_h1 and impressions_h1; I did not check the other folds." |
| "Likely reason it is hard: regression to the mean." | An explanation I did not test. | "One possible explanation, not tested here: pages near the 5-click floor tend to fall back." |
| "Modest, directional improvement over the baseline." | Needs its scope. | "A small, directional improvement over the baseline in one month under a client-grouped split; decision-support for where to look first, not a reliable predictor (ROC-AUC under 0.6)." |

My boldest sentence is the first row. The code cell below prints its rewritten version with the numbers from this run.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
g = res[res.split == GRP].set_index("model")
gap_m = abs(g.loc["logreg", "ROC-AUC"] - g.loc["hist_gb", "ROC-AUC"])
same = gap_m < g.loc["logreg", "fold std"]
print("REWRITTEN CLAIM (numbers from this run):")
print(f"Under a client-grouped split in one month (March 2026), logistic regression reached out-of-fold "
      f"ROC-AUC {g.loc['logreg','ROC-AUC']:.3f} and gradient boosting {g.loc['hist_gb','ROC-AUC']:.3f} "
      f"(logreg fold std about {g.loc['logreg','fold std']:.2f}), against {g.loc[BASE,'ROC-AUC']:.3f} for the baseline. "
      + ("The gap between the two models is smaller than the fold-to-fold spread, so this evidence cannot separate them. "
         if same else "The gap between the two models is larger than the fold-to-fold spread, but this is one month only. ")
      + "Treat the score as directional decision-support for where to look first, not as a reliable predictor.")


REWRITTEN CLAIM (numbers from this run):
Under a client-grouped split in one month (March 2026), logistic regression reached out-of-fold ROC-AUC 0.586 and gradient boosting 0.569 (logreg fold std about 0.04), against 0.549 for the baseline. The gap between the two models is smaller than the fold-to-fold spread, so this evidence cannot separate them. Treat the score as directional decision-support for where to look first, not as a reliable predictor.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.